# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [6]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [7]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [8]:
# Task 1
# YOUR CODE HERE
df['decade'] = (df['release_year'] // 10 * 10).astype('str') + 's'
rating = df[df['rating'].isin(['TV-14','TV-MA','PG-13','R','PG'])]
recent = df.loc[df['added_year'] >= 2010]
rating_year = rating.groupby(['rating', 'decade']).size().reset_index(name='count')
pivot = rating_year.pivot(index='rating', columns='decade', values='count').fillna(0)



# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.imshow(
    pivot,
    color_continuous_scale='Blues',    # sequential one-color
    #aspect='auto',                     # let the chart fill the space naturally
    labels={'color': 'Titles Added'},
    text_auto=True,                    # show count value inside each cell
    height=700, width=1200
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
fig.update_traces(
    textfont=dict(size=9, color='white'),   # white text readable on dark cells
    hovertemplate='<b>%{y}</b> — %{x}<br>Titles: %{z}<extra></extra>',
)

fig.update_layout(
    title='content by rating and release decade',
    font=dict(family='Arial', size=11),
    coloraxis_colorbar=dict(title='Titles Added', thickness=15),
    margin=dict(l=180, r=40, t=55, b=60),
    #coloraxis_showscale=False
)


fig.update_yaxes(title='')

fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [33]:
# Task 2
# YOUR CODE HERE

movie = df.loc[df['type'] == 'Movie']
adds = movie.groupby('added_year').size().reset_index(name='new_titles')
adds = adds.loc[adds['added_year'] >= 2015].copy()
adds = adds.loc[adds['added_year'] <= 2022].copy()

cumulative = adds['new_titles'].sum()

trace = go.Waterfall(
    x=adds['added_year'].astype(str).tolist() + ['Total Added 2015-2022'],  # building x-ticks labels
    y=adds['new_titles'].tolist() + [cumulative],                     # building y-ticks labels
    measure=['relative']*len(adds) + ['total'],   # relative = bars stack; total = final sum
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='#70AD47'),       # green for additions (positive)
    totals=dict(marker_color='#2E75B6'),           # blue for the total bar
    texttemplate='%{y:,}',                         # shows the y values as annotation
    textposition='outside'
)

my_data = [trace]
fig = go.Figure(data=my_data)

fig.update_layout(
    title='2016 and 2019 is the year with the largest single movie addition',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    yaxis=dict(gridcolor='#EEEEEE', title='Titles Added'),
    xaxis=dict(title='Year', showgrid=False),
    margin=dict(l=60, r=40, t=55, b=40),
    showlegend=False,
    height=700
)

fig.add_annotation(
    x= 4 , y=413,
    text= 'This one', showarrow=True, arrowcolor='rgb(200,100,80)', arrowhead=1,
    font=dict(size=20, family='Arial', color='rgb(200,100,80)')
)

fig.add_annotation(
    x= 1 , y=164,
    text= 'And this one', showarrow=True, arrowcolor='rgb(200,100,80)', arrowhead=1,
    font=dict(size=20, family='Arial', color='rgb(200,100,80)')
)

fig.show()